In [58]:
import requests
import torch
from bert_score import BERTScorer as RawBERTScorer
import pandas as pd
from collections import defaultdict

# Host -> services
LEGAL_SERVICE_URL = "http://localhost:8000/api/v1"
DATA_SERVICE_HOST_URL = "http://localhost:8002"

# Container -> data_service (for legal_service to fetch)
DATA_SERVICE_INTERNAL_URL = "http://data_service:8002"

TEST_MAP = {
    #"Lehrlingsbetrieb": "test_lehrlingsausbildung",
    #"Gleichbehandlung ausländischer Anbieter": "test_gleichbehandlung",
    #"Gesamtsumme": "test_gesamtsumme",
    #"Preisspanne": "test_preisspanne",
    #"Keine Preisangabe": "test_keine_preisangabe",
    #"Falsche Rechtsmittelbelehrung": "test_falsche_rechtsmittelbelehrung",
    #"Inkonsistenz Termin": "test_inkonsistenz_termin",
    #"Doppelte Bewertung": "test_doppelte_bewertung",
    #"Verzerrung Bewertung": "test_verzerrung_bewertung",
    #"Keine Gewichtung": "test_keine_gewichtung",
    #"Sachfremde Kriterien": "test_sachfremde_kriterien",
    #"Gerichtsferien": "test_gerichtsferien",
    #"Unberechtigte Referenz": "test_unberechtigte_referenz",
    #"Preisgewichtung": "test_preisgewichtung",
    "Preis nicht als Kriterium": "test_preis_nicht_als_kriterium",
}

# load GT table once
gt_df = pd.read_csv("sample.csv")

# use all project IDs from the GT table
PROJECT_IDS = gt_df["GT table"].astype(int).tolist()

USE_ALL = True # flip this only

if USE_ALL:
    PROJECT_IDS = gt_df["GT table"].astype(int).tolist()
else:
    PROJECT_IDS = PROJECT_IDS = [191846, 169402, 189879, 235296, 258830, 190944, 243602, 205270, 229691, 221633, 9583, 196565, 192494]





In [2]:
def build_version_map():
    resp = requests.get(f"{DATA_SERVICE_HOST_URL}/projects")
    resp.raise_for_status()
    projects = resp.json()["data"]["projects"]
    versions = defaultdict(list)
    for p in projects:
        versions[p["project_id"]].append(p["simap_version"])
    return versions

version_map = build_version_map()

def get_simap_version(project_id):
    vals = version_map.get(project_id, [])
    if not vals:
        return "simap"
    return "simap_v2" if "simap_v2" in vals else vals[0]

def run_tests_for_project(project_id):
    simap_version = get_simap_version(project_id)
    project_data_url = (
        f"{DATA_SERVICE_INTERNAL_URL}/api/v1/projects/{project_id}"
        f"?simap_version={simap_version}"
    )
    payload = {
        "project_data_url": project_data_url,
        "test_codes": list(TEST_MAP.values()),
    }
    resp = requests.post(f"{LEGAL_SERVICE_URL}/tests/run", json=payload)
    if not resp.ok:
        print("STATUS:", resp.status_code)
        print("BODY:", resp.text[:2000])
    resp.raise_for_status()
    return resp.json()





In [3]:
import math, pandas as pd

def logprobs_table(logprobs, max_tokens=50):
    rows = []
    for i, t in enumerate((logprobs or {}).get("content", [])[:max_tokens]):
        lp = t.get("logprob")
        rows.append({
            "i": i,
            "token": repr(t.get("token")),
            "logprob": lp,
            "prob": math.exp(lp) if isinstance(lp, (int, float)) else None
        })
    return pd.DataFrame(rows)


In [5]:
run_tests_for_project(PROJECT_IDS[0])


{'status': 'success',
 'data': {'results': [{'test_code': 'test_preis_nicht_als_kriterium',
    'simap_project_number': None,
    'violation_detected': False,
    'confidence': 0.5560780012425661,
    'error_score': 0.44392199875743393,
    'ue_threshold': 0.715,
    'confidence_scorer': 'whitebox_blackbox',
    'execution_info': {'execution_time_ms': 5901.654005050659,
     'model_used': 'openai/gpt-oss-20b',
     'confidence': 0.5560780012425661,
     'error_score': 0.44392199875743393,
     'ue_threshold': 0.715,
     'confidence_scorer': 'whitebox_blackbox',
     'documents': ['1819_schu_ausschreibung_bkp_211.0_baumeisterarbeiten.pdf'],
     'prompt_tokens': 650,
     'completion_tokens': 388,
     'cost_usd': None,
     'checks_executed': [{'check_code': 'preis_nicht_als_kriterium',
       'version': '1.0.0',
       'result': False,
       'llm_answer': 'Zuerst prüfen wir, ob in der Liste der Zuschlagskriterien ein Begriff vorkommt, der auf Preis hinweist. Die angegebenen Kriterie

In [ ]:
# Single active test runner
# Assumes TEST_MAP has exactly one uncommented test.
# Computes only the selected ensemble scorers:
# - noncontradiction
# - entailment
# - bert_score
# - consistency_and_confidence
# - monte_carlo_probability
# - semantic_density
#
# Output:
# - one parquet checkpoint for the active test
# - one CSV for the active test
# - one QC CSV for the active test
# - one failed_ids JSON for the active test
#
# Strict rule:
# - rows with NaNs are never saved
# - cached NaN rows are removed and recalculated on rerun

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List, Optional
from requests import HTTPError
import json
import re
import time
import traceback

import numpy as np
import pandas as pd
import requests
import torch
from bert_score import BERTScorer as RawBERTScorer
try:
    from ue_gpt20b_support import (
        LOADED_ENV_FILES,
        LLM_BACKEND,
        LLM_MODEL,
        SingleLogprobsScorer,
        build_messages,
        extract_message_text,
        fetch_project_summary,
        llm_chat_with_backoff,
        to_uqlm_logprobs,
    )
except ModuleNotFoundError:
    from uncertainty_estimation.ue_gpt20b_support import (
        LOADED_ENV_FILES,
        LLM_BACKEND,
        LLM_MODEL,
        SingleLogprobsScorer,
        build_messages,
        extract_message_text,
        fetch_project_summary,
        llm_chat_with_backoff,
        to_uqlm_logprobs,
    )

if LOADED_ENV_FILES:
    print("Loaded .env files:", [str(p) for p in LOADED_ENV_FILES])
print("LLM_BACKEND:", LLM_BACKEND)
print("LLM_MODEL:", LLM_MODEL)

# -------------------------
# 0) Validate active TEST_MAP
# -------------------------
if len(TEST_MAP) != 1:
    raise ValueError(
        f"TEST_MAP must contain exactly one active test for this cell. Current TEST_MAP keys: {list(TEST_MAP.keys())}"
    )

ACTIVE_TEST_NAME, ACTIVE_TEST_CODE = next(iter(TEST_MAP.items()))

def _slugify_test_name(name: str) -> str:
    s = (
        name.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue")
            .replace("Ä", "Ae").replace("Ö", "Oe").replace("Ü", "Ue")
            .replace("ß", "ss")
    )
    s = re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
    return s

TEST_SLUG = _slugify_test_name(ACTIVE_TEST_NAME)

# -------------------------
# 1) Config
# -------------------------
BASE_MODEL_TAG = "GPT20B"
RUN_TAG = "6METRICS_PARALLEL_V1"

OUT_DIR = Path(r"data")
CKPT_DIR = OUT_DIR / "_ckpt_single_test_finalensemble_gpt20b_qwen235b"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

ROWS_PATH = CKPT_DIR / f"uqlm_finalensemble_{TEST_SLUG}_{BASE_MODEL_TAG}_{RUN_TAG}.parquet"
FAILED_IDS_PATH = CKPT_DIR / f"uqlm_finalensemble_{TEST_SLUG}_{BASE_MODEL_TAG}_{RUN_TAG}_failed_ids.json"

OUT_CSV_PATH = OUT_DIR / f"uqlm_finalensemble_{TEST_SLUG}_{BASE_MODEL_TAG}_{RUN_TAG}.csv"
QC_CSV_PATH = OUT_DIR / f"uqlm_finalensemble_{TEST_SLUG}_{BASE_MODEL_TAG}_{RUN_TAG}_qc.csv"

PROJECT_IDS_LOCAL = gt_df["GT table"].dropna().astype(int).tolist()

NUM_SAMPLES_LOCAL = int(globals().get("NUM_SAMPLES", 3))
SAMPLING_TEMPERATURE_LOCAL = float(globals().get("SAMPLING_TEMPERATURE", 0.7))
MAX_TOKENS_LOCAL = int(globals().get("MAX_TOKENS", 4000))
NLI_MAX_LENGTH_LOCAL = int(globals().get("NLI_MAX_LENGTH", 12000))
MAX_SAMPLE_TRIES_FACTOR_LOCAL = int(globals().get("MAX_SAMPLE_TRIES_FACTOR", 3))
NLI_MODEL_NAME_LOCAL = str(globals().get("NLI_MODEL_NAME", "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"))
BERT_SCORE_MODEL_TYPE_LOCAL = str(globals().get("BERT_SCORE_MODEL_TYPE", "xlm-roberta-base"))
COSINE_MODEL_NAME_LOCAL = str(globals().get("COSINE_MODEL_NAME", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"))

print("NUM_SAMPLES_LOCAL:", NUM_SAMPLES_LOCAL)
print("NLI_MODEL_NAME_LOCAL:", NLI_MODEL_NAME_LOCAL)
print("BERT_SCORE_MODEL_TYPE_LOCAL:", BERT_SCORE_MODEL_TYPE_LOCAL)
print("COSINE_MODEL_NAME_LOCAL:", COSINE_MODEL_NAME_LOCAL)

SAMPLE_PARALLEL_WORKERS_LOCAL = max(1, int(globals().get("SAMPLE_PARALLEL_WORKERS", NUM_SAMPLES_LOCAL)))
print("SAMPLE_PARALLEL_WORKERS_LOCAL:", SAMPLE_PARALLEL_WORKERS_LOCAL)

MAX_UPSTREAM_RETRIES_LOCAL = 6
BACKOFF_BASE_LOCAL = 1.5
BACKOFF_CAP_LOCAL = 20.0

METHOD_COLS = [
    "noncontradiction",
    "entailment",
    "bert_score",
    "consistency_and_confidence",
    "monte_carlo_probability",
    "semantic_density",
]

FINAL_COLS = [
    "base_model",
    "project_id",
    "test_code",
    "test_name",
    "check_code",
    "gt",
    "pred",
    "match",
    "response",
    "num_samples",
    *METHOD_COLS,
]


def _resolve_simap_version_local(project_id: int) -> str:
    if "get_simap_version" in globals():
        return get_simap_version(project_id)
    host_url = globals().get("DATA_SERVICE_HOST_URL", "http://localhost:8002")
    r = requests.get(f"{host_url}/projects", timeout=60)
    r.raise_for_status()
    projects = (r.json().get("data") or {}).get("projects") or []
    vals = [p.get("simap_version") for p in projects if p.get("project_id") == project_id]
    vals = [v for v in vals if isinstance(v, str) and v]
    if not vals:
        return "simap"
    return "simap_v2" if "simap_v2" in vals else vals[0]

def run_tests_for_project(project_id: int) -> Dict[str, Any]:
    legal_service_url = globals().get("LEGAL_SERVICE_URL", "http://localhost:8000/api/v1")
    data_service_internal_url = globals().get("DATA_SERVICE_INTERNAL_URL", "http://data_service:8002")
    simap_version = _resolve_simap_version_local(project_id)
    project_data_url = (
        f"{data_service_internal_url}/api/v1/projects/{project_id}"
        f"?simap_version={simap_version}"
    )
    payload = {
        "project_data_url": project_data_url,
        "test_codes": [ACTIVE_TEST_CODE],
    }
    r = requests.post(f"{legal_service_url}/tests/run", json=payload, timeout=180)
    if not r.ok:
        print("STATUS:", r.status_code)
        print("BODY:", r.text[:2000])
    r.raise_for_status()
    return r.json()

def run_tests_for_project_with_backoff(project_id: int) -> Dict[str, Any]:
    last_err = None
    for attempt in range(MAX_UPSTREAM_RETRIES_LOCAL):
        try:
            return run_tests_for_project(project_id)
        except HTTPError as e:
            last_err = e
            if not _is_transient_http_error(e):
                raise
            wait = min(BACKOFF_CAP_LOCAL, BACKOFF_BASE_LOCAL * (2 ** attempt)) + np.random.uniform(0, 0.7)
            print(f"run_tests_for_project transient error; retry {attempt+1}/{MAX_UPSTREAM_RETRIES_LOCAL} in {wait:.1f}s")
            time.sleep(wait)
    raise last_err

summary_cache_local: Dict[int, Dict[str, Any]] = {}

def _normalize_nli_label(label: Any) -> str:
    text = str(label or "").strip().lower()
    text = text.replace("_", " ").replace("-", " ")
    aliases = {
        "contradiction": "contradiction",
        "contradictory": "contradiction",
        "neutral": "neutral",
        "entailment": "entailment",
        "entails": "entailment",
    }
    return aliases.get(text, text)

class MultilingualNLIScorer:
    def __init__(self, model_name: str, max_length: int, device: Optional[str] = None):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.model_name = model_name
        self.max_length = max_length
        self.device = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

        raw_id2label = getattr(self.model.config, "id2label", {}) or {}
        self.label_to_index = {
            _normalize_nli_label(label): int(idx)
            for idx, label in raw_id2label.items()
        }
        required = {"contradiction", "neutral", "entailment"}
        missing = sorted(required - set(self.label_to_index))
        if missing:
            raise ValueError(
                f"NLI model {model_name!r} is missing expected labels {missing}; id2label={raw_id2label}"
            )

    def _predict(self, premise: str, hypothesis: str) -> np.ndarray:
        encoded = self.tokenizer(
            premise[: self.max_length],
            hypothesis[: self.max_length],
            truncation=True,
            return_tensors="pt",
        )
        encoded = {name: tensor.to(self.device) for name, tensor in encoded.items()}
        with torch.no_grad():
            logits = self.model(**encoded).logits
        values = logits.detach().cpu().numpy()
        return np.exp(values) / np.exp(values).sum(axis=-1, keepdims=True)

    def evaluate(self, responses: List[str], sampled_responses: List[List[str]]) -> Dict[str, List[float]]:
        contradiction_idx = self.label_to_index["contradiction"]
        entailment_idx = self.label_to_index["entailment"]
        noncontradiction_scores = []
        entailment_scores = []

        for response, candidates in zip(responses, sampled_responses):
            nc_values = []
            ent_values = []
            for candidate in candidates:
                if response == candidate:
                    nc_values.append(1.0)
                    ent_values.append(1.0)
                    continue

                left = self._predict(response, candidate)
                right = self._predict(candidate, response)
                nc = float(((1 - left[:, contradiction_idx]) + (1 - right[:, contradiction_idx]))[0] / 2)
                ent = float(((left[:, entailment_idx] + right[:, entailment_idx]))[0] / 2)
                nc_values.append(nc)
                ent_values.append(ent)

            noncontradiction_scores.append(float(np.mean(nc_values)))
            entailment_scores.append(float(np.mean(ent_values)))

        return {
            "noncontradiction": noncontradiction_scores,
            "entailment": entailment_scores,
        }

class ConfigurableBertScorer:
    def __init__(self, model_type: str, device: Optional[str] = None):
        resolved_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_type = model_type
        self.device = resolved_device
        self.scorer = RawBERTScorer(model_type=model_type, device=resolved_device)

    def evaluate(self, responses: List[str], sampled_responses: List[List[str]], progress_bar: Any = None) -> List[float]:
        scores = []
        for response, candidates in zip(responses, sampled_responses):
            duplicated = [response] * len(candidates)
            _, _, f1 = self.scorer.score(duplicated, candidates)
            scores.append(float(np.mean([float(v) for v in f1])))
        return scores

class ConfigurableCosineScorer:
    def __init__(self, model_name: str, device: Optional[str] = None):
        from sentence_transformers import SentenceTransformer

        resolved_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = model_name
        self.device = resolved_device
        self.model = SentenceTransformer(model_name, trust_remote_code=True, device=resolved_device)

    def evaluate(self, responses: List[str], sampled_responses: List[List[str]], progress_bar: Any = None) -> List[float]:
        scores = []
        for response, candidates in zip(responses, sampled_responses):
            duplicated = [response] * len(candidates)
            emb1 = self.model.encode(duplicated)
            emb2 = self.model.encode(candidates)
            cosine_values = []
            for idx in range(len(emb1)):
                cosine_i = float(np.dot(emb1[idx], emb2[idx]) / (np.linalg.norm(emb1[idx]) * np.linalg.norm(emb2[idx])))
                cosine_values.append(0.5 + cosine_i / 2)
            scores.append(float(np.mean(cosine_values)))
        return scores

MULTILINGUAL_NLI_SCORER = MultilingualNLIScorer(
    model_name=NLI_MODEL_NAME_LOCAL,
    max_length=NLI_MAX_LENGTH_LOCAL,
)
MULTILINGUAL_BERT_SCORER = ConfigurableBertScorer(
    model_type=BERT_SCORE_MODEL_TYPE_LOCAL,
)

# Fix: uqlm's NLI class hardcodes label_mapping=["contradiction","neutral","entailment"],
# which only matches microsoft/deberta-large-mnli. The mDeBERTa-v3 XNLI model uses the
# reversed order (0=entailment, 1=neutral, 2=contradiction), so semantic_density reads
# entailment probs as contradiction probs and collapses to ~0. Patch label_mapping to
# whatever the loaded model's id2label actually says.
from uqlm.scorers.shortform.density import SemanticDensity as _SemanticDensity
from uqlm.nli.nli import NLI as _UqlmNLI

# uqlm's NLI.predict concatenates premise + "[SEP]" + hypothesis and tokenizes
# without truncation. On long German prompts + responses this blows past
# mDeBERTa-v3's 512-token context and causes 100x slowdowns (or silent
# garbage). Replace predict with a token-truncating version that mirrors how
# the user's MultilingualNLIScorer handles input.
if not getattr(_UqlmNLI, "_truncation_patched", False):
    def _patched_nli_predict(self, premise: str, hypothesis: str):
        model_max = getattr(self.model.config, "max_position_embeddings", 512)
        tok_max = getattr(self.tokenizer, "model_max_length", model_max)
        # HF sentinel for "no limit" -- fall back to the position cap.
        if not isinstance(tok_max, int) or tok_max > 1_000_000:
            tok_max = model_max
        encoded = self.tokenizer(
            premise[: self.max_length],
            hypothesis[: self.max_length],
            truncation=True,
            max_length=tok_max,
            return_tensors="pt",
        )
        if self.device:
            encoded = {name: tensor.to(self.device) for name, tensor in encoded.items()}
        logits = self.model(**encoded).logits
        values = logits.detach().cpu().numpy() if self.device else logits.detach().numpy()
        return np.exp(values) / np.exp(values).sum(axis=-1, keepdims=True)
    _UqlmNLI.predict = _patched_nli_predict
    _UqlmNLI._truncation_patched = True

# Cache a single NLI instance across SemanticDensity instantiations so the HF
# model is not reloaded for every project. Keyed on (model, max_length, device)
# so different configs don't collide.
_UQLM_NLI_CACHE: Dict[tuple, Any] = {}

def _cached_setup_nli(self, nli_model_name):
    key = (nli_model_name, getattr(self, "max_length", None), str(getattr(self, "device", None)))
    nli = _UQLM_NLI_CACHE.get(key)
    if nli is None:
        nli = _UqlmNLI(
            nli_model_name=nli_model_name,
            device=getattr(self, "device", None),
            max_length=getattr(self, "max_length", 2000),
            verbose=getattr(self, "verbose", False),
        )
        _UQLM_NLI_CACHE[key] = nli
    # The caller (SampledLogprobsScorer) resets nli.probabilities after __init__
    # returns, so sharing is safe for the per-project probability cache.
    self.nli = nli

if not getattr(_SemanticDensity, "_label_mapping_patched", False):
    _orig_sd_init = _SemanticDensity.__init__
    def _patched_sd_init(self, *args, **kwargs):
        _orig_sd_init(self, *args, **kwargs)
        id2label = getattr(self.nli.model.config, "id2label", None) or {}
        if id2label:
            ordered = [_normalize_nli_label(id2label[i]) for i in sorted(int(k) for k in id2label)]
            required = {"contradiction", "neutral", "entailment"}
            if required.issubset(ordered):
                self.nli.label_mapping = ordered
                self.clusterer.nli.label_mapping = ordered
    _SemanticDensity.__init__ = _patched_sd_init
    _SemanticDensity._setup_nli = _cached_setup_nli
    _SemanticDensity._label_mapping_patched = True

# -------------------------
# 3) Helpers
# -------------------------
def _is_transient_http_error(e: Exception) -> bool:
    if not isinstance(e, HTTPError):
        return False
    resp = getattr(e, "response", None)
    if resp is None:
        return True
    return resp.status_code in {408, 409, 425, 429, 500, 502, 503, 504}

def _clean_text_for_metrics(text: Any) -> str:
    t = "" if text is None else str(text)
    t = t.strip()
    t = re.sub(r"</?s>|<unk>|\[/?INST\]", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def _normalize_api_response_local(text: Any) -> str:
    normalized = ("" if text is None else str(text))
    normalized = (normalized
                  .replace("\u00A0", " ")
                  .replace("\u2009", " ")
                  .replace("\u202F", " ")
                  .replace("\u200B", "")
                  .replace("ï¼š", ":"))
    normalized = re.sub(r":\s{2,}", ": ", normalized)
    normalized = normalized.strip()
    normalized = re.sub(r"```+$", "", normalized).rstrip()
    normalized = normalized.replace("\r\n", "\n").replace("\r", "\n")
    return normalized

def _parse_boolean_result_local(api_response: Any) -> Optional[bool]:
    if api_response is None:
        return None

    trimmed = _normalize_api_response_local(api_response)
    tail = trimmed[-400:]

    try:
        parsed = json.loads(trimmed)
        if isinstance(parsed, dict):
            for key in ["ergebnis", "Ergebnis", "ERGEBNIS", "result"]:
                if key in parsed:
                    result = str(parsed[key]).lower()
                    if result in ("true", "false"):
                        return result == "true"
    except (json.JSONDecodeError, ValueError, TypeError):
        pass

    pattern_strict = re.compile(r"(?:ergebnis):\s*[\"\']?`?(true|false)`?[\"\']?(?:\s|`)*$", re.IGNORECASE)
    match = pattern_strict.search(tail) or pattern_strict.search(trimmed)
    if match:
        return match.group(1).lower() == "true"

    german_true = {"ja", "wahr"}
    german_false = {"nein", "falsch"}
    last_line = trimmed.split("\n")[-1].strip().lower()
    last_token_match = re.search(r"([\wÃ¤Ã¶Ã¼Ã„Ã–ÃœÃŸ]+)[\)\]\>\'\"\.,;:!\s`]*$", last_line)
    if last_token_match:
        token = last_token_match.group(1)
        if token in german_true:
            return True
        if token in german_false:
            return False

    match = re.search(r"[\"\']?`?(true|false)`?[\"\']?[\)\]\>\'\"\.,;:!\s?`]*$", tail, re.IGNORECASE)
    if match:
        return match.group(1).lower() == "true"

    matches = list(re.finditer(r"(?:ergebnis):\s*[\"\']?`?(true|false)`?[\"\']?", trimmed, re.IGNORECASE))
    if matches:
        return matches[-1].group(1).lower() == "true"

    last_line = trimmed.split("\n")[-1].strip()
    match = re.search(r"(true|false)[\)\]\>\'\"\.,;:!\s`]*$", last_line, re.IGNORECASE)
    if match:
        return match.group(1).lower() == "true"

    return None

def _canonical_result_line(text: Any) -> str:
    parsed = _parse_boolean_result_local(text)
    if parsed is None:
        return ""
    return f"ERGEBNIS: {'true' if parsed else 'false'}"

def _slice_logprobs_to_final_result(logprobs_payload: Any, expected_result_line: str = "") -> List[Dict[str, Any]]:
    tokens = to_uqlm_logprobs(logprobs_payload)
    if not tokens:
        return []

    token_texts = [str(tok.get("token") or "") for tok in tokens]
    expected_bool = _parse_boolean_result_local(expected_result_line) if expected_result_line else None
    line_only_re = re.compile(
        r"^[\s\"\'`„“]*ERGEBNIS\s*:\s*(true|false)[\s\"\'`„“\)\]\>\.,;:!]*$",
        re.IGNORECASE,
    )

    for start in range(len(tokens) - 1, -1, -1):
        candidate = ""
        for end in range(start + 1, len(tokens) + 1):
            candidate += token_texts[end - 1]
            normalized = _normalize_api_response_local(candidate)
            if not line_only_re.fullmatch(normalized):
                continue
            parsed = _parse_boolean_result_local(normalized)
            if parsed is None:
                continue
            if expected_bool is not None and parsed != expected_bool:
                continue
            return tokens[start:end]

    return []

def _extract_text_from_chat_response(data: Dict[str, Any]) -> str:
    choices = data.get("choices") or []
    if not choices:
        return ""
    msg = choices[0].get("message") or {}
    content = msg.get("content")
    return content.strip() if isinstance(content, str) else ""

def _get_project_summary_cached(project_id: int) -> Dict[str, Any]:
    if project_id not in summary_cache_local:
        simap_version = _resolve_simap_version_local(project_id)
        summary_cache_local[project_id] = fetch_project_summary(project_id, simap_version)
    return summary_cache_local[project_id]

def _fetch_one_sample(messages: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    data = llm_chat_with_backoff(
        messages=messages,
        temperature=SAMPLING_TEMPERATURE_LOCAL,
        max_tokens=MAX_TOKENS_LOCAL,
        logprobs=True,
    )
    choice = (data.get("choices") or [{}])[0]
    text = _clean_text_for_metrics(extract_message_text(choice))
    if not text:
        return None
    sample_logprobs = to_uqlm_logprobs(choice.get("logprobs"))
    if not sample_logprobs:
        return None
    return {"text": text, "logprobs": sample_logprobs}

def _collect_samples_parallel(messages: List[Dict[str, Any]], project_id: int, check_code: str):
    samples = []
    tries = 0
    max_tries = NUM_SAMPLES_LOCAL * MAX_SAMPLE_TRIES_FACTOR_LOCAL

    while len(samples) < NUM_SAMPLES_LOCAL and tries < max_tries:
        remaining = NUM_SAMPLES_LOCAL - len(samples)
        batch_size = min(SAMPLE_PARALLEL_WORKERS_LOCAL, remaining, max_tries - tries)
        if batch_size <= 0:
            break

        with ThreadPoolExecutor(max_workers=batch_size) as pool:
            futures = [pool.submit(_fetch_one_sample, messages) for _ in range(batch_size)]
            tries += batch_size
            for future in as_completed(futures):
                try:
                    result = future.result()
                except Exception:
                    continue
                if result is None:
                    continue
                samples.append(result)
                if len(samples) >= NUM_SAMPLES_LOCAL:
                    break

    if len(samples) != NUM_SAMPLES_LOCAL:
        raise RuntimeError(
            f"Could not get {NUM_SAMPLES_LOCAL} valid samples for project_id={project_id}, check_code={check_code}"
        )

    sample_texts = [sample["text"] for sample in samples[:NUM_SAMPLES_LOCAL]]
    sample_logprobs_list = [sample["logprobs"] for sample in samples[:NUM_SAMPLES_LOCAL]]
    return sample_texts, sample_logprobs_list

def _ensure_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in FINAL_COLS:
        if c not in df.columns:
            if c in {"project_id", "num_samples"}:
                df[c] = pd.Series(dtype="Int64")
            elif c in {"gt", "pred", "match"}:
                df[c] = pd.Series(dtype="boolean")
            elif c in METHOD_COLS:
                df[c] = np.nan
            else:
                df[c] = ""
    return df[FINAL_COLS]

def _assert_no_nan_output(df: pd.DataFrame, context: str = ""):
    bad_mask = df[FINAL_COLS].isna().any(axis=1)
    if bad_mask.any():
        preview = df.loc[bad_mask, ["project_id", "test_code", "check_code"] + METHOD_COLS].head(10)
        raise RuntimeError(
            f"{context} contains NaNs in required columns for {int(bad_mask.sum())} row(s).\n"
            f"{preview.to_string(index=False)}"
        )

def _load_fail_ids() -> set:
    if not FAILED_IDS_PATH.exists():
        return set()
    try:
        return set(json.loads(FAILED_IDS_PATH.read_text(encoding="utf-8")))
    except Exception:
        return set()

def _save_fail_ids(fail_ids: set):
    FAILED_IDS_PATH.write_text(
        json.dumps(sorted(int(x) for x in fail_ids), indent=2),
        encoding="utf-8",
    )

# -------------------------
# 4) Per-project run for the one active test
# -------------------------
def run_one_project_active_test(project_id: int) -> pd.DataFrame:
    resp = run_tests_for_project_with_backoff(project_id)

    gt_row = gt_df.loc[gt_df["GT table"] == project_id]
    if gt_row.empty:
        raise RuntimeError(f"No GT row for project_id={project_id}")
    gt_row = gt_row.iloc[0]

    results = [r for r in resp["data"]["results"] if str(r["test_code"]) == ACTIVE_TEST_CODE]
    if len(results) != 1:
        raise RuntimeError(
            f"Expected exactly one result for {ACTIVE_TEST_CODE} on project_id={project_id}, got {len(results)}"
        )

    r = results[0]
    gt_val = bool(gt_row[ACTIVE_TEST_NAME])
    pred_val = bool(r.get("violation_detected"))

    records = []
    for c in r.get("execution_info", {}).get("checks_executed", []):
        final_answer = _clean_text_for_metrics(c.get("llm_answer", ""))
        if not final_answer:
            raise RuntimeError(
                f"Missing final answer for project_id={project_id}, check_code={c.get('check_code')}"
            )

        full_logprobs = to_uqlm_logprobs(c.get("logprobs"))
        if not full_logprobs:
            raise RuntimeError(
                f"Missing usable logprobs for project_id={project_id}, check_code={c.get('check_code')}"
            )

        records.append(
            {
                "project_id": int(project_id),
                "test_code": ACTIVE_TEST_CODE,
                "test_name": ACTIVE_TEST_NAME,
                "check_code": str(c["check_code"]),
                "response": final_answer,
                "logprobs": full_logprobs,
                "gt": bool(gt_val),
                "pred": bool(pred_val),
                "match": bool(gt_val == pred_val),
            }
        )

    if not records:
        raise RuntimeError(f"No usable checks for project_id={project_id}, test_code={ACTIVE_TEST_CODE}")

    project_summary = _get_project_summary_cached(project_id)

    responses = []
    logprobs_results = []
    sampled_responses = []
    sampled_logprobs_results = []
    prompt_texts = []

    _t_sampling = time.time()
    for rec in records:
        msgs = build_messages(rec["check_code"], project_summary)
        prompt_text = msgs[-1].get("content", "") if msgs else ""

        sample_texts, sample_logprobs_list = _collect_samples_parallel(
            msgs,
            project_id=project_id,
            check_code=rec["check_code"],
        )

        responses.append(rec["response"])
        logprobs_results.append(rec["logprobs"])
        sampled_responses.append(sample_texts)
        sampled_logprobs_results.append(sample_logprobs_list)
        prompt_texts.append(prompt_text)
    _dt_sampling = time.time() - _t_sampling

    _t_nli = time.time()
    cons_scores = MULTILINGUAL_NLI_SCORER.evaluate(
        responses=responses,
        sampled_responses=sampled_responses,
    )
    _dt_nli = time.time() - _t_nli
    bb_cons = pd.DataFrame(
        {
            "noncontradiction": cons_scores["noncontradiction"],
            "entailment": cons_scores["entailment"],
        }
    )

    _t_bert = time.time()
    bb_bert = pd.DataFrame(
        {
            "bert_score": MULTILINGUAL_BERT_SCORER.evaluate(
                responses=responses,
                sampled_responses=sampled_responses,
                progress_bar=None,
            )
        }
    )
    _dt_bert = time.time() - _t_bert

    from uqlm.white_box.sampled_logprobs import SampledLogprobsScorer
    slp = SampledLogprobsScorer(
        scorers=["consistency_and_confidence", "monte_carlo_probability", "semantic_density"],
        nli_model_name=NLI_MODEL_NAME_LOCAL,
        sentence_transformer=COSINE_MODEL_NAME_LOCAL,
        max_length=NLI_MAX_LENGTH_LOCAL,
        length_normalize=True,
        prompts_in_nli=False,
    )
    _t_slp = time.time()
    sl_scores = slp.evaluate(
        responses=responses,
        sampled_responses=sampled_responses,
        logprobs_results=logprobs_results,
        sampled_logprobs_results=sampled_logprobs_results,
        prompts=prompt_texts,
    )
    _dt_slp = time.time() - _t_slp
    print(
        f"    timings pid={project_id}: "
        f"sampling={_dt_sampling:.1f}s | nli={_dt_nli:.1f}s | "
        f"bert={_dt_bert:.1f}s | slp={_dt_slp:.1f}s | "
        f"total={_dt_sampling + _dt_nli + _dt_bert + _dt_slp:.1f}s"
    )
    sampled_df = pd.DataFrame(
        {
            "consistency_and_confidence": sl_scores["consistency_and_confidence"],
            "monte_carlo_probability": sl_scores["monte_carlo_probability"],
            "semantic_density": sl_scores["semantic_density"],
        }
    )

    method_df = pd.concat(
        [
            bb_cons[["noncontradiction"]].reset_index(drop=True),
            bb_cons[["entailment"]].reset_index(drop=True),
            bb_bert[["bert_score"]].reset_index(drop=True),
            sampled_df[["consistency_and_confidence"]].reset_index(drop=True),
            sampled_df[["monte_carlo_probability"]].reset_index(drop=True),
            sampled_df[["semantic_density"]].reset_index(drop=True),
        ],
        axis=1,
    )

    method_df = method_df.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

    if method_df[METHOD_COLS].isna().any().any():
        nan_counts = method_df[METHOD_COLS].isna().sum()
        raise RuntimeError(
            f"NaNs in computed methods for project_id={project_id}: "
            f"{nan_counts[nan_counts > 0].to_dict()}"
        )

    base_df = pd.DataFrame(
        {
            "base_model": BASE_MODEL_TAG,
            "project_id": [r["project_id"] for r in records],
            "test_code": [r["test_code"] for r in records],
            "test_name": [r["test_name"] for r in records],
            "check_code": [r["check_code"] for r in records],
            "gt": [r["gt"] for r in records],
            "pred": [r["pred"] for r in records],
            "match": [r["match"] for r in records],
            "response": [r["response"] for r in records],
            "num_samples": NUM_SAMPLES_LOCAL,
        }
    )

    out = pd.concat([base_df.reset_index(drop=True), method_df.reset_index(drop=True)], axis=1)
    out = _ensure_schema(out)
    _assert_no_nan_output(out, context=f"run_one_project_active_test({project_id})")
    return out

# -------------------------
# 5) Restore checkpoint
# -------------------------
if ROWS_PATH.exists():
    try:
        final_df = pd.read_parquet(ROWS_PATH)
    except Exception:
        final_df = pd.DataFrame(columns=FINAL_COLS)
else:
    final_df = pd.DataFrame(columns=FINAL_COLS)

final_df = _ensure_schema(final_df)

recalc_ids = set()
if not final_df.empty:
    bad_mask = final_df[FINAL_COLS].isna().any(axis=1)
    if bad_mask.any():
        recalc_ids = set(final_df.loc[bad_mask, "project_id"].dropna().astype(int).tolist())
        final_df = final_df.loc[~final_df["project_id"].isin(recalc_ids)].copy()
        final_df.to_parquet(ROWS_PATH, index=False)

done_ids = set(final_df["project_id"].dropna().astype(int).unique().tolist()) if not final_df.empty else set()
failed_ids = _load_fail_ids()

pending_ids = [
    pid for pid in PROJECT_IDS_LOCAL
    if (pid not in done_ids) or (pid in recalc_ids) or (pid in failed_ids)
]

print("Active test:", ACTIVE_TEST_NAME, f"[{ACTIVE_TEST_CODE}]")
print("Checkpoint:", ROWS_PATH)
print("Cached rows:", len(final_df))
print("Done projects:", len(done_ids))
print("Recalc IDs from cached NaNs:", len(recalc_ids))
print("Queued projects:", len(pending_ids))

# -------------------------
# 6) Run active test
# -------------------------
for i, pid in enumerate(pending_ids, start=1):
    print(f"[{i}/{len(pending_ids)}] project_id={pid}")
    try:
        out = run_one_project_active_test(int(pid))

        final_df = final_df.loc[final_df["project_id"] != int(pid)].copy()
        final_df = pd.concat([final_df, out], ignore_index=True)
        final_df = _ensure_schema(final_df)
        _assert_no_nan_output(final_df, context=f"checkpoint after pid={pid}")

        final_df.to_parquet(ROWS_PATH, index=False)

        if int(pid) in failed_ids:
            failed_ids.remove(int(pid))
            _save_fail_ids(failed_ids)

    except Exception as e:
        failed_ids.add(int(pid))
        _save_fail_ids(failed_ids)
        print(f"FAILED project_id={pid}: {e}")
        print(traceback.format_exc(limit=2))

# -------------------------
# 7) Finalize + export
# -------------------------
final_df = _ensure_schema(final_df)
_assert_no_nan_output(final_df, context="final_df")

final_df = final_df.sort_values(["project_id", "check_code"]).reset_index(drop=True)
final_df.to_parquet(ROWS_PATH, index=False)
final_df.to_csv(OUT_CSV_PATH, index=False)

qc_df = pd.DataFrame(
    [{
        "test_name": ACTIVE_TEST_NAME,
        "test_code": ACTIVE_TEST_CODE,
        "rows": int(len(final_df)),
        "failed_projects": len(failed_ids),
        "nan_any_required": int(final_df[FINAL_COLS].isna().any(axis=1).sum()) if not final_df.empty else 0,
        **{f"nan_{c}": int(final_df[c].isna().sum()) if not final_df.empty else 0 for c in METHOD_COLS},
    }]
)
qc_df.to_csv(QC_CSV_PATH, index=False)

print("\nSaved table:")
print(OUT_CSV_PATH)
print("\nSaved QC:")
print(QC_CSV_PATH)
print("\nFailed project IDs:")
print(sorted(failed_ids))


In [74]:
final_df

,base_model,project_id,test_code,test_name,check_code,gt,pred,match,response,num_samples,noncontradiction,entailment,bert_score,consistency_and_confidence,monte_carlo_probability,semantic_density
0,GPT20B,9583,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,Zuerst lese ich den gesamten Text des Feldes „...,3,0.999850,0.997694,0.903759,0.887460,0.861236,0.994084
1,GPT20B,169402,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,Zuerst habe ich die Liste der Schlüsselbegriff...,3,0.999762,0.997447,0.890847,0.854002,0.781257,0.997195
2,GPT20B,189879,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,Zuerst lese ich den Inhalt des Feldes award_cr...,3,0.709000,0.520353,0.888468,0.909604,0.823335,0.991409
3,GPT20B,190944,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,Schritt 1: Der Text im Feld award_criteria ent...,3,0.999769,0.995836,0.895533,0.921020,0.764501,0.992998
4,GPT20B,191846,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,Zuerst wird der Inhalt des Feldes „award_crite...,3,0.999876,0.994408,0.901395,0.913569,0.800331,0.986703
5,GPT20B,192494,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,"Zunächst prüfen wir, ob in den angegebenen Kri...",3,0.999733,0.888521,0.885527,0.888860,0.816893,0.998284
6,GPT20B,196565,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,Zunächst wird der Text des Feldes „award_crite...,3,0.999811,0.833467,0.893512,0.843416,0.801532,0.992331
7,GPT20B,205270,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,"Zuerst prüfen wir, ob in den angegebenen Krite...",3,0.999751,0.992476,0.903356,0.938355,0.762134,0.989431
8,GPT20B,221633,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,False,False,True,Zuerst habe ich den Text des Feldes „award_cri...,3,0.999897,0.997882,0.904489,0.868919,0.706456,0.991576
9,GPT20B,229691,test_preis_nicht_als_kriterium,Preis nicht als Kriterium,preis_nicht_als_kriterium,True,True,True,Zunächst wird der Inhalt des Feldes „award_cri...,3,0.999853,0.833766,0.950434,0.957334,0.780736,0.988600


In [76]:
final_df.to_csv("GPT20_Ensemble5_Preis_nicht_als_Kriterium.csv", index=False)

In [ ]:

# ---------------------------------------------------------------------------
# All-tests uncertainty table: 15 tests x all required checks x 13 GT ids.
#
# Fast path:
# - call legal_service once per project with all 15 test codes
# - sample each unique project/check only once
# - reuse that check-level uncertainty across every test row that contains it
#
# Answers saved per row:
# - answer_1_original from legal_service
# - answer_2_sample ... answer_5_sample from 4 fresh LLM samples
# - prompt, system_prompt, original/sample logprobs JSON
#
# Metrics computed, no judge:
# - semantic_density
# - bert_score
# - noncontradiction
# - entailment
# ---------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List, Optional
from requests import HTTPError
import json
import re
import time
import traceback

import numpy as np
import pandas as pd
import requests
import torch
from bert_score import BERTScorer as RawBERTScorer

try:
    from ue_gpt20b_support import (
        LOADED_ENV_FILES,
        LLM_BACKEND,
        LLM_MODEL,
        build_messages,
        extract_message_text,
        fetch_project_summary,
        llm_chat_with_backoff,
        to_uqlm_logprobs,
    )
except ModuleNotFoundError:
    from uncertainty_estimation.ue_gpt20b_support import (
        LOADED_ENV_FILES,
        LLM_BACKEND,
        LLM_MODEL,
        build_messages,
        extract_message_text,
        fetch_project_summary,
        llm_chat_with_backoff,
        to_uqlm_logprobs,
    )

print("Loaded .env files:", [str(p) for p in LOADED_ENV_FILES])
print("LLM_BACKEND:", LLM_BACKEND)
print("LLM_MODEL:", LLM_MODEL)

# -------------------------
# 0) Inputs and config
# -------------------------
OUT_DIR = Path(r"data")
CKPT_DIR = OUT_DIR / "_ckpt_epic6_all15_gpt20b_4samples"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_TAG = "GPT20B"
RUN_TAG = "EPIC6_ALL15_4SAMPLES_NOJUDGE_V1"
ROWS_PATH = CKPT_DIR / f"uqlm_all15_{BASE_MODEL_TAG}_{RUN_TAG}.parquet"
FAILED_IDS_PATH = CKPT_DIR / f"uqlm_all15_{BASE_MODEL_TAG}_{RUN_TAG}_failed_ids.json"
OUT_CSV_PATH = OUT_DIR / f"uqlm_all15_{BASE_MODEL_TAG}_{RUN_TAG}.csv"
QC_CSV_PATH = OUT_DIR / f"uqlm_all15_{BASE_MODEL_TAG}_{RUN_TAG}_qc.csv"

# These are exactly the 15 GT columns in Epic6_GT.csv.
ALL_TEST_MAP = {
    "Lehrlingsbetrieb": "test_lehrlingsausbildung",
    "Gleichbehandlung ausländischer Anbieter": "test_gleichbehandlung",
    "Gesamtsumme": "test_gesamtsumme",
    "Preisspanne": "test_preisspanne",
    "Keine Preisangabe": "test_keine_preisangabe",
    "Falsche Rechtsmittelbelehrung": "test_falsche_rechtsmittelbelehrung",
    "Inkonsistenz Termin": "test_inkonsistenz_termin",
    "Doppelte Bewertung": "test_doppelte_bewertung",
    "Verzerrung Bewertung": "test_verzerrung_bewertung",
    "Keine Gewichtung": "test_keine_gewichtung",
    "Sachfremde Kriterien": "test_sachfremde_kriterien",
    "Gerichtsferien": "test_gerichtsferien",
    "Unberechtigte Referenz": "test_unberechtigte_referenz",
    "Preisgewichtung": "test_preisgewichtung",
    "Preis nicht als Kriterium": "test_preis_nicht_als_kriterium",
}
TEST_CODE_TO_NAME = {v: k for k, v in ALL_TEST_MAP.items()}

GT_TABLE_PATH = OUT_DIR / "Epic6_GT.csv"
if not GT_TABLE_PATH.exists():
    raise FileNotFoundError(f"Could not find Epic6 GT table: {GT_TABLE_PATH}")

# Always reload the requested Epic6 GT table so earlier notebook cells cannot
# leave a stale sample.csv-backed gt_df in memory.
gt_df = pd.read_csv(GT_TABLE_PATH)
print("GT table:", GT_TABLE_PATH)

missing_gt_cols = [name for name in ALL_TEST_MAP if name not in gt_df.columns]
if missing_gt_cols:
    raise ValueError(f"Missing GT columns in Epic6_GT.csv: {missing_gt_cols}")

PROJECT_IDS_LOCAL = gt_df["GT table"].dropna().astype(int).tolist()
if len(PROJECT_IDS_LOCAL) != 13:
    raise ValueError(f"Expected 13 project IDs from Epic6_GT.csv, got {len(PROJECT_IDS_LOCAL)}")

NUM_SAMPLES_LOCAL = int(globals().get("NUM_SAMPLES", 4))
if NUM_SAMPLES_LOCAL != 4:
    print(f"Overriding NUM_SAMPLES={NUM_SAMPLES_LOCAL} to 4 for this run.")
    NUM_SAMPLES_LOCAL = 4
SAMPLING_TEMPERATURE_LOCAL = float(globals().get("SAMPLING_TEMPERATURE", 0.7))
MAX_TOKENS_LOCAL = int(globals().get("MAX_TOKENS", 4000))
MAX_SAMPLE_TRIES_FACTOR_LOCAL = int(globals().get("MAX_SAMPLE_TRIES_FACTOR", 3))
SAMPLE_PARALLEL_WORKERS_LOCAL = max(1, int(globals().get("SAMPLE_PARALLEL_WORKERS", NUM_SAMPLES_LOCAL)))
NLI_MAX_LENGTH_LOCAL = int(globals().get("NLI_MAX_LENGTH", 12000))
NLI_MODEL_NAME_LOCAL = str(globals().get("NLI_MODEL_NAME", "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"))
BERT_SCORE_MODEL_TYPE_LOCAL = str(globals().get("BERT_SCORE_MODEL_TYPE", "xlm-roberta-base"))
COSINE_MODEL_NAME_LOCAL = str(globals().get("COSINE_MODEL_NAME", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"))

print("Projects:", PROJECT_IDS_LOCAL)
print("Tests:", len(ALL_TEST_MAP))
print("Fresh samples per check:", NUM_SAMPLES_LOCAL)
print("SAMPLE_PARALLEL_WORKERS_LOCAL:", SAMPLE_PARALLEL_WORKERS_LOCAL)
print("NLI_MODEL_NAME_LOCAL:", NLI_MODEL_NAME_LOCAL)
print("BERT_SCORE_MODEL_TYPE_LOCAL:", BERT_SCORE_MODEL_TYPE_LOCAL)

METHOD_COLS = ["semantic_density", "bert_score", "noncontradiction", "entailment"]
ANSWER_COLS = ["answer_1_original", "answer_2_sample", "answer_3_sample", "answer_4_sample", "answer_5_sample"]
FINAL_COLS = [
    "base_model", "project_id", "test_code", "test_name", "check_code", "gt", "pred", "match",
    "check_pred", "check_parse_ok", "check_result_line",
    "prompt", "system_prompt", *ANSWER_COLS, "all_answers_json", "num_samples",
    "original_logprobs_json", "sample_logprobs_json", *METHOD_COLS,
]

# -------------------------
# 1) Metric helpers
# -------------------------
def _normalize_nli_label(label: Any) -> str:
    text = str(label or "").strip().lower().replace("_", " ").replace("-", " ")
    aliases = {
        "contradiction": "contradiction",
        "contradictory": "contradiction",
        "neutral": "neutral",
        "entailment": "entailment",
        "entails": "entailment",
    }
    return aliases.get(text, text)

class MultilingualNLIScorer:
    def __init__(self, model_name: str, max_length: int, device: Optional[str] = None):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.model_name = model_name
        self.max_length = max_length
        self.device = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        raw_id2label = getattr(self.model.config, "id2label", {}) or {}
        self.label_to_index = {_normalize_nli_label(label): int(idx) for idx, label in raw_id2label.items()}
        missing = sorted({"contradiction", "neutral", "entailment"} - set(self.label_to_index))
        if missing:
            raise ValueError(f"NLI model {model_name!r} missing labels {missing}; id2label={raw_id2label}")

    def _predict(self, premise: str, hypothesis: str) -> np.ndarray:
        encoded = self.tokenizer(
            str(premise)[: self.max_length],
            str(hypothesis)[: self.max_length],
            truncation=True,
            return_tensors="pt",
        )
        encoded = {name: tensor.to(self.device) for name, tensor in encoded.items()}
        with torch.no_grad():
            logits = self.model(**encoded).logits
        values = logits.detach().cpu().numpy()
        values = values - values.max(axis=-1, keepdims=True)
        return np.exp(values) / np.exp(values).sum(axis=-1, keepdims=True)

    def evaluate(self, responses: List[str], sampled_responses: List[List[str]]) -> Dict[str, List[float]]:
        contradiction_idx = self.label_to_index["contradiction"]
        entailment_idx = self.label_to_index["entailment"]
        noncontradiction_scores = []
        entailment_scores = []
        for response, candidates in zip(responses, sampled_responses):
            nc_values = []
            ent_values = []
            for candidate in candidates:
                if response == candidate:
                    nc_values.append(1.0)
                    ent_values.append(1.0)
                    continue
                left = self._predict(response, candidate)
                right = self._predict(candidate, response)
                nc = float(((1 - left[:, contradiction_idx]) + (1 - right[:, contradiction_idx]))[0] / 2)
                ent = float(((left[:, entailment_idx] + right[:, entailment_idx]))[0] / 2)
                nc_values.append(nc)
                ent_values.append(ent)
            noncontradiction_scores.append(float(np.mean(nc_values)))
            entailment_scores.append(float(np.mean(ent_values)))
        return {"noncontradiction": noncontradiction_scores, "entailment": entailment_scores}

class ConfigurableBertScorer:
    def __init__(self, model_type: str, device: Optional[str] = None):
        resolved_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.scorer = RawBERTScorer(model_type=model_type, device=resolved_device)

    def evaluate(self, responses: List[str], sampled_responses: List[List[str]]) -> List[float]:
        scores = []
        for response, candidates in zip(responses, sampled_responses):
            duplicated = [response] * len(candidates)
            _, _, f1 = self.scorer.score(duplicated, candidates)
            scores.append(float(np.mean([float(v) for v in f1])))
        return scores

def _patch_uqlm_semantic_density():
    from uqlm.scorers.shortform.density import SemanticDensity as _SemanticDensity
    from uqlm.nli.nli import NLI as _UqlmNLI

    if not getattr(_UqlmNLI, "_truncation_patched", False):
        def _patched_nli_predict(self, premise: str, hypothesis: str):
            model_max = getattr(self.model.config, "max_position_embeddings", 512)
            tok_max = getattr(self.tokenizer, "model_max_length", model_max)
            if not isinstance(tok_max, int) or tok_max > 1_000_000:
                tok_max = model_max
            encoded = self.tokenizer(
                str(premise)[: self.max_length],
                str(hypothesis)[: self.max_length],
                truncation=True,
                max_length=tok_max,
                return_tensors="pt",
            )
            if self.device:
                encoded = {name: tensor.to(self.device) for name, tensor in encoded.items()}
            logits = self.model(**encoded).logits
            values = logits.detach().cpu().numpy() if self.device else logits.detach().numpy()
            values = values - values.max(axis=-1, keepdims=True)
            return np.exp(values) / np.exp(values).sum(axis=-1, keepdims=True)
        _UqlmNLI.predict = _patched_nli_predict
        _UqlmNLI._truncation_patched = True

    if "_UQLM_NLI_CACHE_ALL15" not in globals():
        globals()["_UQLM_NLI_CACHE_ALL15"] = {}

    def _cached_setup_nli(self, nli_model_name):
        cache = globals()["_UQLM_NLI_CACHE_ALL15"]
        key = (nli_model_name, getattr(self, "max_length", None), str(getattr(self, "device", None)))
        nli = cache.get(key)
        if nli is None:
            nli = _UqlmNLI(
                nli_model_name=nli_model_name,
                device=getattr(self, "device", None),
                max_length=getattr(self, "max_length", 2000),
                verbose=getattr(self, "verbose", False),
            )
            cache[key] = nli
        self.nli = nli

    if not getattr(_SemanticDensity, "_all15_label_mapping_patched", False):
        _orig_sd_init = _SemanticDensity.__init__
        def _patched_sd_init(self, *args, **kwargs):
            _orig_sd_init(self, *args, **kwargs)
            id2label = getattr(self.nli.model.config, "id2label", None) or {}
            if id2label:
                ordered = [_normalize_nli_label(id2label[i]) for i in sorted(int(k) for k in id2label)]
                if {"contradiction", "neutral", "entailment"}.issubset(ordered):
                    self.nli.label_mapping = ordered
                    self.clusterer.nli.label_mapping = ordered
        _SemanticDensity.__init__ = _patched_sd_init
        _SemanticDensity._setup_nli = _cached_setup_nli
        _SemanticDensity._all15_label_mapping_patched = True

_patch_uqlm_semantic_density()
from uqlm.white_box.sampled_logprobs import SampledLogprobsScorer

MULTILINGUAL_NLI_SCORER = globals().get("MULTILINGUAL_NLI_SCORER") or MultilingualNLIScorer(
    model_name=NLI_MODEL_NAME_LOCAL,
    max_length=NLI_MAX_LENGTH_LOCAL,
)
MULTILINGUAL_BERT_SCORER = globals().get("MULTILINGUAL_BERT_SCORER") or ConfigurableBertScorer(
    model_type=BERT_SCORE_MODEL_TYPE_LOCAL,
)
SEMANTIC_DENSITY_SCORER = SampledLogprobsScorer(
    scorers=["semantic_density"],
    nli_model_name=NLI_MODEL_NAME_LOCAL,
    sentence_transformer=COSINE_MODEL_NAME_LOCAL,
    max_length=NLI_MAX_LENGTH_LOCAL,
    length_normalize=True,
    prompts_in_nli=False,
)

# -------------------------
# 2) Service / sample helpers
# -------------------------
def _json_dumps(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False)

def _clean_text_for_metrics(text: Any) -> str:
    t = "" if text is None else str(text)
    t = re.sub(r"</?s>|<unk>|\[/?INST\]", " ", t.strip())
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

_RESULT_RE_LOCAL = re.compile(r"""\bERGEBNIS\s*:\s*['"`]?\s*(true|false)\s*['"`]?""", re.IGNORECASE)

def _normalize_api_response_local(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for key in ("llm_answer", "content", "text", "output_text"):
            candidate = value.get(key)
            if isinstance(candidate, str):
                return candidate
        choices = value.get("choices")
        if isinstance(choices, list) and choices:
            first = choices[0] or {}
            message = first.get("message") if isinstance(first, dict) else None
            if isinstance(message, dict) and isinstance(message.get("content"), str):
                return message["content"]
            if isinstance(first, dict) and isinstance(first.get("text"), str):
                return first["text"]
    return str(value)


def _parse_boolean_result_local(value: Any) -> bool | None:
    text = _normalize_api_response_local(value)
    matches = list(_RESULT_RE_LOCAL.finditer(text))
    if matches:
        return matches[-1].group(1).lower() == "true"
    for line in reversed([ln.strip() for ln in text.splitlines() if ln.strip()]):
        compact = line.strip("'\"` .")
        if compact.lower() in {"true", "false"}:
            return compact.lower() == "true"
    return None


def _canonical_result_line(value: Any) -> str:
    parsed = _parse_boolean_result_local(value)
    if parsed is None:
        return ""
    return f"ERGEBNIS: {str(parsed).lower()}"

def _is_transient_http_error(e: Exception) -> bool:
    if not isinstance(e, HTTPError):
        return False
    resp = getattr(e, "response", None)
    if resp is None:
        return True
    return resp.status_code in {408, 409, 425, 429, 500, 502, 503, 504}

def _resolve_simap_version_all15(project_id: int) -> str:
    if "get_simap_version" in globals():
        return get_simap_version(project_id)
    host_url = globals().get("DATA_SERVICE_HOST_URL", "http://localhost:8002")
    r = requests.get(f"{host_url}/projects", timeout=60)
    r.raise_for_status()
    projects = (r.json().get("data") or {}).get("projects") or []
    vals = [p.get("simap_version") for p in projects if p.get("project_id") == project_id]
    vals = [v for v in vals if isinstance(v, str) and v]
    return "simap_v2" if "simap_v2" in vals else (vals[0] if vals else "simap")

def _fetch_project_summary_cached(project_id: int) -> Dict[str, Any]:
    cache = globals().setdefault("_ALL15_PROJECT_SUMMARY_CACHE", {})
    if project_id not in cache:
        simap_version = _resolve_simap_version_all15(project_id)
        cache[project_id] = fetch_project_summary(project_id, simap_version)
    return cache[project_id]

def run_all_tests_for_project(project_id: int) -> Dict[str, Any]:
    legal_service_url = globals().get("LEGAL_SERVICE_URL", "http://localhost:8000/api/v1")
    data_service_internal_url = globals().get("DATA_SERVICE_INTERNAL_URL", "http://data_service:8002")
    simap_version = _resolve_simap_version_all15(project_id)
    project_data_url = f"{data_service_internal_url}/api/v1/projects/{project_id}?simap_version={simap_version}"
    payload = {"project_data_url": project_data_url, "test_codes": list(ALL_TEST_MAP.values())}
    r = requests.post(f"{legal_service_url}/tests/run", json=payload, timeout=300)
    if not r.ok:
        print("STATUS:", r.status_code)
        print("BODY:", r.text[:3000])
    r.raise_for_status()
    return r.json()

def run_all_tests_for_project_with_backoff(project_id: int) -> Dict[str, Any]:
    last_err = None
    for attempt in range(6):
        try:
            return run_all_tests_for_project(project_id)
        except HTTPError as e:
            last_err = e
            if not _is_transient_http_error(e):
                raise
            wait = min(30.0, 1.5 * (2 ** attempt)) + np.random.uniform(0, 0.7)
            print(f"run_all_tests transient error; retry {attempt + 1}/6 in {wait:.1f}s")
            time.sleep(wait)
    raise last_err

def _fetch_one_sample(messages: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    data = llm_chat_with_backoff(
        messages=messages,
        temperature=SAMPLING_TEMPERATURE_LOCAL,
        max_tokens=MAX_TOKENS_LOCAL,
        logprobs=True,
    )
    choice = (data.get("choices") or [{}])[0]
    text = _clean_text_for_metrics(extract_message_text(choice))
    if not text:
        return None
    sample_logprobs = to_uqlm_logprobs(choice.get("logprobs"))
    if not sample_logprobs:
        return None
    return {"text": text, "logprobs": sample_logprobs}

def _collect_samples_parallel(messages: List[Dict[str, Any]], project_id: int, check_code: str):
    samples = []
    tries = 0
    max_tries = NUM_SAMPLES_LOCAL * MAX_SAMPLE_TRIES_FACTOR_LOCAL
    while len(samples) < NUM_SAMPLES_LOCAL and tries < max_tries:
        remaining = NUM_SAMPLES_LOCAL - len(samples)
        batch_size = min(SAMPLE_PARALLEL_WORKERS_LOCAL, remaining, max_tries - tries)
        with ThreadPoolExecutor(max_workers=batch_size) as pool:
            futures = [pool.submit(_fetch_one_sample, messages) for _ in range(batch_size)]
            tries += batch_size
            for future in as_completed(futures):
                try:
                    result = future.result()
                except Exception:
                    continue
                if result is not None:
                    samples.append(result)
                if len(samples) >= NUM_SAMPLES_LOCAL:
                    break
    if len(samples) != NUM_SAMPLES_LOCAL:
        raise RuntimeError(f"Could not get {NUM_SAMPLES_LOCAL} samples for project_id={project_id}, check_code={check_code}")
    return [s["text"] for s in samples], [s["logprobs"] for s in samples]

def _metric_record_for_check(project_id: int, check_code: str, check_payload: Dict[str, Any]) -> Dict[str, Any]:
    original_answer_raw = _normalize_api_response_local(check_payload.get("llm_answer", ""))
    original_answer = _clean_text_for_metrics(original_answer_raw)
    if not original_answer:
        raise RuntimeError(f"Missing answer for project_id={project_id}, check_code={check_code}")

    parsed_check = _parse_boolean_result_local(original_answer_raw)
    if parsed_check is None:
        tail = original_answer[-500:]
        raise RuntimeError(
            f"Could not parse final ERGEBNIS line for project_id={project_id}, "
            f"check_code={check_code}. answer_tail={tail!r}"
        )
    check_result_line = _canonical_result_line(original_answer_raw)

    original_logprobs = to_uqlm_logprobs(check_payload.get("logprobs"))
    if not original_logprobs:
        raise RuntimeError(f"Missing logprobs for project_id={project_id}, check_code={check_code}")
    if any(str(tok.get("token", "")).startswith("<|") for tok in original_logprobs if isinstance(tok, dict)):
        raise RuntimeError(f"Control tokens leaked into parsed logprobs for project_id={project_id}, check_code={check_code}")

    prompt = str(check_payload.get("prompt") or "")
    system_prompt = str(check_payload.get("system_prompt") or "")
    if prompt and system_prompt:
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]
    else:
        project_summary = _fetch_project_summary_cached(project_id)
        messages = build_messages(check_code, project_summary)
        system_prompt = messages[0].get("content", "") if messages else system_prompt
        prompt = messages[-1].get("content", "") if messages else prompt

    sample_texts, sample_logprobs = _collect_samples_parallel(messages, project_id, check_code)
    responses = [original_answer]
    sampled_responses = [sample_texts]
    logprobs_results = [original_logprobs]
    sampled_logprobs_results = [sample_logprobs]

    t0 = time.time()
    nli_scores = MULTILINGUAL_NLI_SCORER.evaluate(responses, sampled_responses)
    bert_scores = MULTILINGUAL_BERT_SCORER.evaluate(responses, sampled_responses)
    sd_scores = SEMANTIC_DENSITY_SCORER.evaluate(
        responses=responses,
        sampled_responses=sampled_responses,
        logprobs_results=logprobs_results,
        sampled_logprobs_results=sampled_logprobs_results,
        prompts=[prompt],
    )
    print(f"    scored project_id={project_id}, check={check_code} in {time.time() - t0:.1f}s")

    all_answers = [original_answer, *sample_texts]
    return {
        "check_pred": bool(parsed_check),
        "check_parse_ok": True,
        "check_result_line": check_result_line,
        "prompt": prompt,
        "system_prompt": system_prompt,
        "answer_1_original": original_answer,
        "answer_2_sample": sample_texts[0],
        "answer_3_sample": sample_texts[1],
        "answer_4_sample": sample_texts[2],
        "answer_5_sample": sample_texts[3],
        "all_answers_json": _json_dumps(all_answers),
        "num_samples": NUM_SAMPLES_LOCAL,
        "original_logprobs_json": _json_dumps(original_logprobs),
        "sample_logprobs_json": _json_dumps(sample_logprobs),
        "semantic_density": float(sd_scores["semantic_density"][0]),
        "bert_score": float(bert_scores[0]),
        "noncontradiction": float(nli_scores["noncontradiction"][0]),
        "entailment": float(nli_scores["entailment"][0]),
    }

def _ensure_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in FINAL_COLS:
        if col not in df.columns:
            if col == "project_id" or col == "num_samples":
                df[col] = pd.Series(dtype="Int64")
            elif col in {"gt", "pred", "match", "check_pred", "check_parse_ok"}:
                df[col] = pd.Series(dtype="boolean")
            elif col in METHOD_COLS:
                df[col] = np.nan
            else:
                df[col] = ""
    return df[FINAL_COLS]

def _project_is_complete(df: pd.DataFrame, project_id: int, expected_rows: int) -> bool:
    if df.empty or "project_id" not in df.columns:
        return False
    part = df.loc[df["project_id"].astype("Int64") == int(project_id)].copy()
    if len(part) != expected_rows:
        return False
    required = ["test_code", "test_name", "check_code", "check_pred", "check_parse_ok", "check_result_line", "prompt", *ANSWER_COLS, *METHOD_COLS]
    if part[required].isna().any().any():
        return False
    for col in ["prompt", *ANSWER_COLS]:
        if (part[col].astype(str).str.len() == 0).any():
            return False
    return True

def _load_failed_ids() -> set:
    if not FAILED_IDS_PATH.exists():
        return set()
    try:
        return set(json.loads(FAILED_IDS_PATH.read_text(encoding="utf-8")))
    except Exception:
        return set()

def _save_failed_ids(failed_ids: set):
    FAILED_IDS_PATH.write_text(json.dumps(sorted(int(x) for x in failed_ids), indent=2), encoding="utf-8")

# -------------------------
# 3) Per-project table builder
# -------------------------
def run_one_project_all_tests(project_id: int) -> pd.DataFrame:
    response = run_all_tests_for_project_with_backoff(project_id)
    gt_row = gt_df.loc[gt_df["GT table"].astype(int) == int(project_id)]
    if gt_row.empty:
        raise RuntimeError(f"No GT row for project_id={project_id}")
    gt_row = gt_row.iloc[0]

    results = response.get("data", {}).get("results", [])
    results_by_code = {str(r.get("test_code")): r for r in results}
    missing = [code for code in ALL_TEST_MAP.values() if code not in results_by_code]
    if missing:
        raise RuntimeError(f"Missing legal_service results for project_id={project_id}: {missing}")

    unique_checks: Dict[str, Dict[str, Any]] = {}
    row_refs = []
    for test_name, test_code in ALL_TEST_MAP.items():
        result = results_by_code[test_code]
        gt_val = bool(gt_row[test_name])
        pred_val = bool(result.get("violation_detected"))
        checks = result.get("execution_info", {}).get("checks_executed", [])
        if not checks:
            raise RuntimeError(f"No checks for project_id={project_id}, test_code={test_code}")
        for check in checks:
            check_code = str(check.get("check_code"))
            unique_checks.setdefault(check_code, check)
            row_refs.append({
                "base_model": BASE_MODEL_TAG,
                "project_id": int(project_id),
                "test_code": test_code,
                "test_name": test_name,
                "check_code": check_code,
                "gt": gt_val,
                "pred": pred_val,
                "match": bool(gt_val == pred_val),
            })

    print(f"  unique checks to sample/score: {len(unique_checks)}; table rows: {len(row_refs)}")
    metric_by_check = {}
    for idx, (check_code, check_payload) in enumerate(unique_checks.items(), start=1):
        print(f"  [{idx}/{len(unique_checks)}] check={check_code}")
        metric_by_check[check_code] = _metric_record_for_check(project_id, check_code, check_payload)

    rows = []
    for row in row_refs:
        rows.append({**row, **metric_by_check[row["check_code"]]})
    out = _ensure_schema(pd.DataFrame(rows))

    method_values = out[METHOD_COLS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if method_values.isna().any().any():
        raise RuntimeError(f"NaN metric values for project_id={project_id}: {method_values.isna().sum().to_dict()}")
    for col in METHOD_COLS:
        out[col] = method_values[col]
    return out

# Expected rows per project from legal_service test definitions, excluding tests not in sample.csv.
EXPECTED_ROWS_PER_PROJECT = 25
EXPECTED_TOTAL_ROWS = EXPECTED_ROWS_PER_PROJECT * len(PROJECT_IDS_LOCAL)

# -------------------------
# 4) Restore checkpoint and run missing projects
# -------------------------
if ROWS_PATH.exists():
    try:
        final_df = pd.read_parquet(ROWS_PATH)
    except Exception:
        final_df = pd.DataFrame(columns=FINAL_COLS)
else:
    final_df = pd.DataFrame(columns=FINAL_COLS)
final_df = _ensure_schema(final_df)

failed_ids = _load_failed_ids()
pending_ids = [
    int(pid) for pid in PROJECT_IDS_LOCAL
    if (not _project_is_complete(final_df, int(pid), EXPECTED_ROWS_PER_PROJECT)) or (int(pid) in failed_ids)
]

print("Checkpoint:", ROWS_PATH)
print("Cached rows:", len(final_df))
print("Expected total rows:", EXPECTED_TOTAL_ROWS)
print("Pending projects:", pending_ids)

for i, pid in enumerate(pending_ids, start=1):
    print(f"\n[{i}/{len(pending_ids)}] project_id={pid}")
    try:
        project_df = run_one_project_all_tests(pid)
        final_df = final_df.loc[final_df["project_id"].astype("Int64") != int(pid)].copy()
        final_df = pd.concat([final_df, project_df], ignore_index=True)
        final_df = _ensure_schema(final_df)
        final_df = final_df.sort_values(["project_id", "test_code", "check_code"]).reset_index(drop=True)
        final_df.to_parquet(ROWS_PATH, index=False)
        final_df.to_csv(OUT_CSV_PATH, index=False)
        if int(pid) in failed_ids:
            failed_ids.remove(int(pid))
            _save_failed_ids(failed_ids)
    except Exception as e:
        failed_ids.add(int(pid))
        _save_failed_ids(failed_ids)
        print(f"FAILED project_id={pid}: {e}")
        print(traceback.format_exc(limit=3))

# -------------------------
# 5) Finalize, save, display full table
# -------------------------
final_df = _ensure_schema(final_df)
final_df = final_df.loc[final_df["project_id"].astype("Int64").isin(PROJECT_IDS_LOCAL)].copy()
final_df = final_df.sort_values(["project_id", "test_code", "check_code"]).reset_index(drop=True)
final_df.to_parquet(ROWS_PATH, index=False)
final_df.to_csv(OUT_CSV_PATH, index=False)

qc_df = pd.DataFrame([{
    "rows": int(len(final_df)),
    "expected_rows": int(EXPECTED_TOTAL_ROWS),
    "projects": int(final_df["project_id"].nunique()) if not final_df.empty else 0,
    "expected_projects": int(len(PROJECT_IDS_LOCAL)),
    "tests": int(final_df["test_code"].nunique()) if not final_df.empty else 0,
    "expected_tests": int(len(ALL_TEST_MAP)),
    "failed_projects": _json_dumps(sorted(int(x) for x in failed_ids)),
    **{f"nan_{col}": int(final_df[col].isna().sum()) if col in final_df else None for col in METHOD_COLS},
}])
qc_df.to_csv(QC_CSV_PATH, index=False)

print("\nSaved full table:", OUT_CSV_PATH)
print("Saved checkpoint:", ROWS_PATH)
print("Saved QC:", QC_CSV_PATH)
print("Failed project IDs:", sorted(failed_ids))
print(f"Rows: {len(final_df)} / {EXPECTED_TOTAL_ROWS}")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 240)

# Keep final_df full fidelity for downstream use and saved files, but avoid
# rendering megabytes of token logprob JSON in the notebook output.
display_df = final_df.copy()
for _col in ["original_logprobs_json", "sample_logprobs_json"]:
    display_df[_col] = display_df[_col].astype(str).str.slice(0, 500) + " ..."
display(display_df)


Loaded .env files: ['..\\intelliprocure-ai-legal\\legal_service\\.env', '.env']
LLM_BACKEND: together
LLM_MODEL: openai/gpt-oss-20b
GT table: C:\Users\Feresh\Documents\thesis\uncertainty_estimation\Epic6_GT.csv
Projects: [191846, 169402, 189879, 235296, 258830, 190944, 243602, 205270, 229691, 221633, 9583, 196565, 192494]
Tests: 15
Fresh samples per check: 4
SAMPLE_PARALLEL_WORKERS_LOCAL: 4
NLI_MODEL_NAME_LOCAL: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
BERT_SCORE_MODEL_TYPE_LOCAL: xlm-roberta-base
Checkpoint: C:\Users\Feresh\Documents\thesis\uncertainty_estimation\_ckpt_epic6_all15_gpt20b_4samples\uqlm_all15_GPT20B_EPIC6_ALL15_4SAMPLES_NOJUDGE_V1.parquet
Cached rows: 325
Expected total rows: 325
Pending projects: []

Saved full table: C:\Users\Feresh\Documents\thesis\uncertainty_estimation\uqlm_all15_GPT20B_EPIC6_ALL15_4SAMPLES_NOJUDGE_V1.csv
Saved checkpoint: C:\Users\Feresh\Documents\thesis\uncertainty_estimation\_ckpt_epic6_all15_gpt20b_4samples\uqlm_all15_GPT20B_EPIC6_ALL15_4SAMPLES

,base_model,project_id,test_code,test_name,check_code,gt,pred,match,check_pred,check_parse_ok,check_result_line,prompt,system_prompt,answer_1_original,answer_2_sample,answer_3_sample,answer_4_sample,answer_5_sample,all_answers_json,num_samples,original_logprobs_json,sample_logprobs_json,semantic_density,bert_score,noncontradiction,entailment
0,GPT20B,9583,test_doppelte_bewertung,Doppelte Bewertung,redundant_criteria,False,False,True,False,True,ERGEBNIS: false,"AUFGABE:\n\nPrüfe: Wird ein Kriterium, welches unter ""award_criteria"" gelistet ist, auch in ""selection_criteria"" oder ""technical_specifications"" gelistet? \nFalls ja, gib ""ERGEBNIS: true"" zurück, ansonsten ""ERGEBNIS: false"".\n\nAchten S...","Führe folgende 'AUFGABE' aus und erkläre deine Schlussfolgerung kurz, max 1000 Token.Die Instruktionen gelten sinngemäss für andere Sprachen als Deutsch.Wenn keine Informationen vorhanden sind, antworte mit 'ERGEBNIS: false'Wenn das Fel...","Zuerst prüfe ich, ob in den Feldern „selection_criteria“ oder „technical_specifications“ Informationen vorhanden sind. \nIm bereitgestellten Text steht explizit: \n- **selection_criteria:** Keine Information gefunden. \n- **technical_sp...","Zuerst prüfen wir, welche Informationen in den Feldern „selection_criteria“ und „technical_specifications“ vorhanden sind. \nFür beide Felder wurde angegeben: „Keine Information gefunden.“ Das bedeutet, dass keine Kriterien, Anforderung...","Schritt 1: Die Datenfelder werden geprüft. \nDie Felder „selection_criteria“ und „technical_specifications“ enthalten jeweils den Text “Keine Information gefunden.”, d. h. sie sind leer. \n\nSchritt 2: Das Feld „award_criteria“ listet m...","Schritt 1: Die Liste der „award_criteria“ enthält die Kriterien für die Zuschlagskriterien des Projekts (Preis, Fachkompetenz Bestattungen, Fachkompetenz Unterhalt, Soziales Engagement). \nSchritt 2: Die Angabe zu „selection_criteria“ l...",Zuerst habe ich die Inhalte der drei Felder geprüft. \nFür **selection_criteria** wurde explizit angegeben: *Keine Information gefunden.* \nFür **technical_specifications** wurde ebenfalls angegeben: *Keine Information gefunden.* \nDas ...,"[""Zuerst prüfe ich, ob in den Feldern „selection_criteria“ oder „technical_specifications“ Informationen vorhanden sind. \nIm bereitgestellten Text steht explizit: \n- **selection_criteria:** Keine Information gefunden. \n- **technical_...",4,"[{""token"": ""Z"", ""bytes"": [90], ""logprob"": 0.0, ""top_logprobs"": [{""token"": ""Z"", ""bytes"": [90], ""logprob"": 0.0}, {""token"": ""Sch"", ""bytes"": [83, 99, 104], ""logprob"": -35.0}, {""token"": ""Analyse"", ""bytes"": [65, 110, 97, 108, 121, 115, 101], ...","[[{""token"": ""Z"", ""bytes"": [90], ""logprob"": -0.08466929942369461, ""top_logprobs"": [{""token"": ""Z"", ""bytes"": [90], ""logprob"": -0.08466929942369461}, {""token"": ""Um"", ""bytes"": [85, 109], ""logprob"": -3.656099557876587}, {""token"": ""Sch"", ""byte...",0.999195,0.900431,0.999759,0.993236
1,GPT20B,9583,test_falsche_rechtsmittelbelehrung,Falsche Rechtsmittelbelehrung,frist_nicht_20T,False,False,True,False,True,ERGEBNIS: false,"AUFGABE:\nAnalysieren Sie die ""instructions_on_legal_remedies"" und prüfen Sie, ob eine Frist angegeben wird, diese aber nicht 20 Tage beträgt. \nAntworten Sie mit ""true"" wenn eine Frist angegeben ist, diese aber nicht 20 Tage beträgt. W...","Führe folgende 'AUFGABE' aus und erkläre deine Schlussfolgerung kurz, max 1000 Token.Die Instruktionen gelten sinngemäss für andere Sprachen als Deutsch.Wenn keine Informationen vorhanden sind, antworte mit 'ERGEBNIS: false'Wenn das Fel...",Zuerst lese ich den Text in der Variable instructions_on_legal_remedies. \nDer Satz lautet: „Der Zuschlagsentscheid kann innert 20 Tagen seit der Publikation … angefochten werden.“ \nDamit wird eine Frist genannt – die Frist beträgt gen...,"Die Passage enthält die Formulierung „innert 20 Tagen“. \nDies ist eine klare Angabe einer Frist. \nDie Frist beträgt exakt 20 Tage, nicht

In [21]:
final_df.to_csv("GPT20_Ensemble_all_v5.csv", index=False)

In [ ]:
# Recompute BERTScore with a larger multilingual model for GPT20_Ensemble_all_v1..v5.
# Previous combined files used xlm-roberta-base in the active bert_score column.
# This cell leaves bert_score untouched and writes the large-model values to
# bert_score_xlm_roberta_large so calibration can compare latency/quality tradeoffs.

from pathlib import Path
import os
import sys
from typing import List

import numpy as np
import pandas as pd
import torch
from bert_score import BERTScorer as RawBERTScorer

DATA_DIR = Path(globals().get("DATA_DIR", r"data"))
COMBINED_ENSEMBLE_FILES = [
    DATA_DIR / "GPT20_Ensemble_all_v1.csv",
    DATA_DIR / "GPT20_Ensemble_all_v2.csv",
    DATA_DIR / "GPT20_Ensemble_all_v3.csv",
    DATA_DIR / "GPT20_Ensemble_all_v4.csv",
    DATA_DIR / "GPT20_Ensemble_all_v5.csv",
]

# xlm-roberta-large is multilingual and stronger than xlm-roberta-base.
BERT_SCORE_MODEL_TYPE = "xlm-roberta-large"
BERT_SCORE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BERT_SCORE_BATCH_SIZE = 16 if BERT_SCORE_DEVICE == "cuda" else 4
WRITE_BACK = True

ANSWER_COLS = [
    "answer_1_original",
    "answer_2_sample",
    "answer_3_sample",
    "answer_4_sample",
    "answer_5_sample",
]
BASE_COL = "bert_score"
LARGE_COL = "bert_score_xlm_roberta_large"
MODEL_COL = "bert_score_xlm_roberta_large_model_type"

print("BERT_SCORE_MODEL_TYPE:", BERT_SCORE_MODEL_TYPE)
print("BERT_SCORE_DEVICE:", BERT_SCORE_DEVICE)
print("BERT_SCORE_BATCH_SIZE:", BERT_SCORE_BATCH_SIZE)

bert_scorer_large = RawBERTScorer(
    model_type=BERT_SCORE_MODEL_TYPE,
    device=BERT_SCORE_DEVICE,
)

def _clean_answer(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def compute_large_bert_scores(df: pd.DataFrame) -> np.ndarray:
    missing = [c for c in ANSWER_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing answer columns: {missing}")

    candidates: List[str] = []
    references: List[str] = []
    row_ids: List[int] = []
    for row_idx, row in df.iterrows():
        original = _clean_answer(row["answer_1_original"])
        for sample_col in ANSWER_COLS[1:]:
            sample = _clean_answer(row[sample_col])
            if original and sample:
                candidates.append(original)
                references.append(sample)
                row_ids.append(row_idx)

    if not candidates:
        return np.full(len(df), np.nan, dtype=float)

    _, _, f1 = bert_scorer_large.score(
        candidates,
        references,
        batch_size=BERT_SCORE_BATCH_SIZE,
        verbose=True,
    )

    sums = np.zeros(len(df), dtype=float)
    counts = np.zeros(len(df), dtype=int)
    index_to_pos = {idx: pos for pos, idx in enumerate(df.index)}
    for row_idx, score in zip(row_ids, f1.detach().cpu().numpy()):
        pos = index_to_pos[row_idx]
        sums[pos] += float(score)
        counts[pos] += 1

    out = np.full(len(df), np.nan, dtype=float)
    mask = counts > 0
    out[mask] = sums[mask] / counts[mask]
    return out

summary_rows = []
for path in COMBINED_ENSEMBLE_FILES:
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    if BASE_COL not in df.columns:
        raise ValueError(f"{path.name}: missing {BASE_COL}")

    base_scores = df[BASE_COL].astype(float).copy()
    large_scores = compute_large_bert_scores(df)
    df[LARGE_COL] = large_scores
    df[MODEL_COL] = BERT_SCORE_MODEL_TYPE

    changed = np.nanmean(np.abs(base_scores.to_numpy(dtype=float) - large_scores))
    corr = pd.Series(base_scores).corr(pd.Series(large_scores))
    summary_rows.append({
        "file": path.name,
        "rows": len(df),
        "base_mean": float(np.nanmean(base_scores)),
        "large_mean": float(np.nanmean(large_scores)),
        "mean_abs_change": float(changed),
        "base_large_corr": float(corr) if pd.notna(corr) else np.nan,
        "large_col": LARGE_COL,
        "write_back": WRITE_BACK,
    })

    if WRITE_BACK:
        df.to_csv(path, index=False)

bert_recompute_summary = pd.DataFrame(summary_rows)
display(bert_recompute_summary.round(6))
print("Done. Existing bert_score was left unchanged.")
print("To test the large model in calibration, include/use column:", LARGE_COL)


BERT_SCORE_MODEL_TYPE: xlm-roberta-large
BERT_SCORE_DEVICE: cpu
BERT_SCORE_BATCH_SIZE: 4
calculating scores...
computing bert embedding.


  0%|          | 0/292 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/325 [00:00<?, ?it/s]

done in 282.86 seconds, 4.60 sentences/sec
calculating scores...
computing bert embedding.


...

computing greedy matching.


  0%|          | 0/325 [00:00<?, ?it/s]

done in 277.73 seconds, 4.68 sentences/sec


,file,rows,base_mean,large_mean,mean_abs_change,base_large_corr,large_col,write_back
0,GPT20_Ensemble_all_v1.csv,325,0.910990,0.918059,0.007232,0.977775,bert_score_xlm_roberta_large,True
1,GPT20_Ensemble_all_v2.csv,325,0.911659,0.918302,0.006643,0.981218,bert_score_xlm_roberta_large,True
2,GPT20_Ensemble_all_v3.csv,325,0.912042,0.918954,0.006952,0.978910,bert_score_xlm_roberta_large,True
3,GPT20_Ensemble_all_v4.csv,325,0.912431,0.918933,0.006520,0.975399,bert_score_xlm_roberta_large,True
4,GPT20_Ensemble_all_v5.csv,325,0.912982,0.919116,0.006340,0.975983,bert_score_xlm_roberta_large,True


Done. Existing bert_score was left unchanged.
To test the large model in calibration, include/use column: bert_score_xlm_roberta_large
